# Q1 유지 · Q2/Q3 독립 검증 (수작업 없음)

**새 L4 런타임 → Secret `GITHUB_TOKEN` 액세스 허용 → 런타임 / 모두 실행.** Drive 인증을 한 번 허용하면 박스 그리기·정답 입력 없이 진행됩니다. 이전에 실행한 dev 기준선과 원본 ZIP을 자동으로 찾습니다.

기본: InternVL, 같은 dev 40장. **Q1은 기존 답을 그대로 유지하며 신규 호출 0회**입니다. Q2/Q3에는 같은 정상 참조(diffdate), 대상 이미지, greedy 추론을 씁니다.

| 조건 | Q2 | Q3 |
|---|---|---|
| baseline | 기존 답 | 새 직접 질문 |
| q2_define | 자기 답 + 결함 정의로 재검토 | 새 직접 질문 |
| q2_define_q3_independent | 자기 답 + 결함 정의로 재검토 | 영역 유무 독립 질문 → 필요할 때 전체 이미지 위치 질문 |
| matched_reask | 자기 답으로 단순 재질문 | 직접 질문 반복, agent와 이미지별 호출 수 동일 |

Q2는 자기 Q2 답만, Q3는 이미지와 자기 질문만 씁니다. Q1 답으로 Q2/Q3를 덮어쓰지 않습니다. 같은 모델의 정의/유무/위치 도구를 사용하며, 외부 vision expert는 없습니다.

**Q3 자동 채점 범위:** dev에는 위치 정답이 없습니다. 기존 정상/결함 라벨로 `영역 없음 / 영역 있음`만 자동 채점합니다. 결함 이미지에서 맞는 위치를 찾았는지는 **미검증**이며 `q3_location_accuracy = null`입니다. Q3의 9구역+없음(10보기)은 기존 gold 4보기와 다른 보조 실험입니다. 전체 결함이 시각적으로 국소화 가능하다는 가정도 위치 정답으로 검증하지 않았습니다.

단순히 전부 '없음'이라고 답해 좋아 보이는 현상을 확인하도록 항상 없음/항상 영역 대조군과 정상/결함 균형 정확도를 함께 출력합니다. Q1+Q2+Q3 joint는 **Q3 영역 유무만 포함**합니다.

신규 호출: InternVL 총 **200~280회** (40장, Q3가 영역 있음이면 추가 위치 호출). 실제 GPU 시간은 아직 측정하지 않았습니다. 20호출마다 Drive 저장. 중단되면 추론 셀 재실행으로 성공한 답을 재사용하고 실패한 호출을 재시도합니다. 실패가 남은 조건은 비교표에서 제외하고 셀을 중단합니다. 무효 답은 따로 기록합니다.

출력: `MyDrive/lvlm_insplad_results/gold_v2_q2_q3_independent_dev_v1/`. 기존 dev/gold 결과와 frozen 설정은 보존합니다. 설정 선택·gold 실행·결과 GitHub 게시를 자동으로 하지 않습니다. 입력/모델/seed를 바꾸면 RUN_FOLDER를 새 이름으로 바꾸세요. 마지막 셀에서 결과 ZIP을 자동 다운로드합니다.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_q2_q3_independent_dev_v1"
MODELS = ["internvl3_5_8b"]  # 필요하면 "qwen3_5_4b" 추가 (호출 2배)
N_IMAGES = 40
SEED = 42
CHECKPOINT_EVERY = 20
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
if not RUN_ROOT.name.startswith("gold_v2_q2_q3_independent_dev"):
    raise ValueError("새 실험 전용 RUN_FOLDER를 사용하세요.")
DEV_REFS_DIR = DEV_DIR / "refs"
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"

REPO_DIR = Path("/content/lvlm_q2_q3_agent_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
if not REPO_DIR.exists():
    run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
else:
    existing_remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if existing_remote != f"https://github.com/{REPO}.git":
        raise RuntimeError("실행 폴더의 저장소가 다릅니다. 새 런타임을 사용하세요.")
    # A cached checkout is kept; do not silently change its code during resume.
    if not (REPO_DIR / "scripts/run_gold_v2_q2_q3_agent_dev.py").is_file():
        raise RuntimeError("이 런타임은 이전 코드입니다. 새 런타임을 사용하세요.")
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError("VRAM 20GB 이상 L4/A100이 필요합니다.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"dev 세트가 FROZEN이 아닙니다: {summary.get('quality_status')}")
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_REFS_DIR / "retrieved_refs.csv"
need = [DEV_QA, REFS_CSV] + [STAGE1_DIR / m / "normal_ref_retrieved_diffdate.jsonl" for m in MODELS]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("기존 dev 기준선 입력 없음: " + ", ".join(missing))
print(f"입력 확인 완료 | {N_IMAGES}장, Q1 재호출 0회, Q2/Q3 신규 호출 {N_IMAGES*5*len(MODELS)}~{N_IMAGES*7*len(MODELS)}회")


In [ ]:
# 추론 및 자동 평가. 중단되면 이 셀 재실행 (실패한 호출만 재시도)
RUNNER_PATH = REPO_DIR / "scripts/run_gold_v2_q2_q3_agent_dev.py"
if not RUNNER_PATH.is_file():
    raise FileNotFoundError("merge된 main을 사용하세요. 새 런타임에서 다시 열면 됩니다.")
run([sys.executable, RUNNER_PATH, "--dev-qa", DEV_QA, "--stage1-dir", STAGE1_DIR,
     "--refs-csv", REFS_CSV, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--models", *MODELS, "--n-images", str(N_IMAGES),
     "--seed", str(SEED), "--checkpoint-every", str(CHECKPOINT_EVERY)])


In [ ]:
# 자동 결과 확인
pd.set_option("display.max_rows", 200)
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
meta = json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
print("실패 호출:", meta["row_errors"], "이번 신규 호출:", meta["called_this_run"], "재사용:", meta["reused_calls"])
display(pd.DataFrame(status))
if meta["row_errors"]:
    raise RuntimeError("실패가 남았습니다. 추론 셀을 다시 실행하세요. 실패 조건은 비교 대상에서 제외됩니다.")
answers = pd.read_csv(RUN_ROOT / "answers.csv")
q1 = answers[answers.question_type == "anomaly_detection"]
if q1.empty:
    raise RuntimeError("Q1 답이 없습니다.")
assert (q1.predicted_choice == q1.cached_baseline_choice).all(), "Q1 답 변경!"
print("Q1 기존 답 유지 확인:", len(q1), "행 모두 동일, 추론 0회")
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(lb[["model_key", "policy", "q1_anomaly_f1", "q2_defect_macro_f1", "q2_accuracy",
            "joint_accuracy", "false_defect_rate", "new_q2_q3_calls"]].round(3))
print("Q3 아래 점수는 영역 유무 보조 지표이며 위치 정확도가 아닙니다.")
display(lb[["model_key", "policy", "q3_presence_balanced_accuracy", "q3_normal_none_accuracy",
            "q3_normal_false_region_rate", "q3_anomaly_false_none_rate", "q3_anomaly_region_recall",
            "q3_invalid_rate", "q1_q2_q3_presence_joint"]].round(3))
print("q3_location_accuracy:", meta["q3_location_accuracy"], "(미검증 / 위치 정답 없음)")
print("결함 종류별 Q2:")
display(pd.read_csv(RUN_ROOT / "q2_by_defect.csv").round(3))
print("항상 없음 / 항상 영역 대조군:")
display(pd.read_csv(RUN_ROOT / "q3_constant_baselines.csv").round(3))
print("이미지별 paired bootstrap CI (탐색용): matched_reask 대비도 확인")
display(pd.read_csv(RUN_ROOT / "significance.csv").round(3))
print("결과 위치:", RUN_ROOT)


In [ ]:
# 결과 ZIP 다운로드 (Drive 결과는 그대로 유지)
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/dev_q2_q3_agent_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in RUN_ROOT.rglob("*"):
        if p.is_file() and "_images" not in p.relative_to(RUN_ROOT).parts:
            z.write(p, p.relative_to(RUN_ROOT))
files.download(str(archive))
